# ResQ AI — Emergency Priority Prediction

This notebook builds a practical, hackathon-ready ML pipeline that predicts **operational emergency priority** (`CRITICAL` / `HIGH` / `MEDIUM` / `LOW`) for a disaster incident.

**What this model is (and isn't):**
- It estimates *operational urgency* — where should responders act first, given people affected, casualties, vulnerable population, and resource strain.
- It does **not** decide whose life is more valuable. It never sees or uses any feature that ranks individual people.
- ResQ AI helps responders identify incidents where **delay may have greater operational consequences**, so limited ambulances, rescue teams, shelters and medical supplies can be routed faster to where the gap between demand and capacity is largest.

Sections:
1. Data preparation (load + merge)
2. Target investigation & leakage check
3. Preprocessing
4. Model training (Experiment A vs Experiment B)
5. Model comparison & evaluation
6. Feature importance
7. Cross-validation
8. Sample predictions & explanation
9. Saving the model


## 1. Data preparation — load & merge

In [55]:
import pandas as pd
import numpy as np
import joblib
import os

pd.set_option('display.max_columns', 60)

DATA_DIR = "../data/raw"   # change if your paths differ

disaster_df = pd.read_csv(f"{DATA_DIR}/disaster_dataset_cleaned.csv")
area_df     = pd.read_csv(f"{DATA_DIR}/areas.csv")
incident_df = pd.read_csv(f"{DATA_DIR}/incident_dataset.csv")
resource_df = pd.read_csv(f"{DATA_DIR}/resources.csv")

print("disaster_df:", disaster_df.shape)
print("area_df    :", area_df.shape)
print("incident_df:", incident_df.shape)
print("resource_df:", resource_df.shape)


disaster_df: (432, 37)
area_df    : (785, 18)
incident_df: (5184, 33)
resource_df: (33757, 11)


In [56]:
# incident_df is our central ML dataset
incident_df.head()


,incident_id,disaster_id,area_id,state,district,disaster_type,disaster_subtype,population,people_affected,deaths,injured,critical_patients,children_affected,elderly_affected,hospital_count,ambulance_count,available_ambulances,rescue_team_count,available_rescue_teams,shelter_capacity,available_shelter_capacity,hospital_capacity,medical_supply,ambulance_demand,rescue_team_demand,shelter_demand,medical_supply_demand,resource_shortage,severity_score,vulnerable_ratio,priority_score,priority_level,data_status
0,INC000001,2026-0330-IND,A0753,Jammu and Kashmir,Budgam,Storm,Lightning/Thunderstorms,1017803,1289,93,65,15,148,124,21,35,23,8,7,84713,84713,5448,1901,4,1,593,421,0.0012,0.8170,0.2110,26.38,MEDIUM,synthetic_scenario
1,INC000002,2026-0330-IND,A0469,Odisha,Sundargarh,Storm,Lightning/Thunderstorms,264399,750,52,26,6,64,74,5,5,4,2,1,25626,25626,1133,717,2,1,342,101,0.0023,0.8680,0.1840,25.70,MEDIUM,synthetic_scenario
2,INC000003,2026-0330-IND,A0439,Nagaland,Zunheboto,Storm,Lightning/Thunderstorms,211214,1623,121,102,18,207,209,3,7,5,1,1,25191,25191,1120,425,5,1,914,786,0.0780,0.7204,0.2563,27.36,MEDIUM,synthetic_scenario
3,INC000004,2026-0330-IND,A0706,Uttarakhand,Rudraprayag,Storm,Lightning/Thunderstorms,235236,1358,124,68,21,182,82,3,5,3,2,2,12094,12094,553,1092,6,1,852,440,0.1623,0.8983,0.1944,31.09,MEDIUM,synthetic_scenario
4,INC000005,2026-0330-IND,A0152,Chhattisgarh,Raipur,Storm,Lightning/Thunderstorms,326039,1529,128,96,15,200,106,7,14,10,1,0,40328,40328,2028,1196,4,1,826,430,0.2047,0.7531,0.2001,27.54,MEDIUM,synthetic_scenario


### 1.1 Merge `incident_df` with `area_df` on `area_id`

`incident_df` already carries area-level operational counts (`hospital_count`, `ambulance_count`,
`shelter_capacity`, `rescue_team_count`, `population`, `state`, `district`) that are **identical**
to the corresponding columns in `area_df` (verified below) — they were clearly generated from the
same source. We only bring in the *new* demographic columns from `area_df` to avoid duplicate
columns after the merge.

In [57]:
overlap_cols = ['state', 'district', 'population', 'hospital_count',
                'ambulance_count', 'shelter_capacity', 'rescue_team_count', 'data_status']

# sanity check: confirm these really are identical before dropping them from area_df
check = incident_df.merge(area_df, on='area_id', suffixes=('_inc', '_area'))
for c in ['population', 'hospital_count', 'ambulance_count', 'shelter_capacity', 'rescue_team_count']:
    match_rate = (check[f"{c}_inc"] == check[f"{c}_area"]).mean()
    print(f"{c}: {match_rate:.1%} identical between incident_df and area_df")

area_extra = area_df.drop(columns=[c for c in overlap_cols if c in area_df.columns])
df = incident_df.merge(area_extra, on='area_id', how='left')
print(df.shape)


population: 100.0% identical between incident_df and area_df
hospital_count: 100.0% identical between incident_df and area_df
ambulance_count: 100.0% identical between incident_df and area_df
shelter_capacity: 100.0% identical between incident_df and area_df
rescue_team_count: 100.0% identical between incident_df and area_df
(5184, 42)


### 1.2 Aggregate `resource_df` by `area_id`

`resource_df` is a resource-level log. Aggregated counts of ambulances/rescue teams/hospitals per
area turn out to be **exactly** what `incident_df`'s `ambulance_count`, `rescue_team_count`, etc.
already contain (verified below) — so re-deriving them would be a duplicate, unnecessary feature.

What `resource_df` *does* add that isn't anywhere else: **resource condition/readiness**
(e.g. how many resources are flagged "Needs Maintenance") and **availability at the individual
resource level**. We add a small number of genuinely new features only:

- `resource_pct_available` — share of an area's resources currently available
- `resource_pct_needs_maintenance` — share of resources flagged as needing maintenance (a readiness signal not present anywhere else)
- `resource_avg_capacity` — average capacity per resource unit in the area


In [58]:
# Verify redundancy before deciding what (not) to re-derive
amb_from_resources = (resource_df[resource_df.resource_type == 'Ambulance']
                       .groupby('area_id').size())
check_amb = incident_df[['area_id', 'ambulance_count']].drop_duplicates('area_id').set_index('area_id')
check_amb['from_resource_df'] = amb_from_resources
match_rate = (check_amb['ambulance_count'] == check_amb['from_resource_df']).mean()
print(f"ambulance_count already matches resource_df aggregation for {match_rate:.1%} of areas -> redundant, skip re-deriving")

res_agg = resource_df.groupby('area_id').agg(
    resource_total_count=('resource_id', 'count'),
    resource_available_count=('availability_status', lambda s: (s == 'Available').sum()),
    resource_needs_maintenance_count=('condition', lambda s: (s == 'Needs Maintenance').sum()),
    resource_avg_capacity=('capacity', 'mean'),
).reset_index()

res_agg['resource_pct_available'] = res_agg['resource_available_count'] / res_agg['resource_total_count']
res_agg['resource_pct_needs_maintenance'] = res_agg['resource_needs_maintenance_count'] / res_agg['resource_total_count']
res_agg = res_agg[['area_id', 'resource_pct_available', 'resource_pct_needs_maintenance', 'resource_avg_capacity']]

df = df.merge(res_agg, on='area_id', how='left')
print(df.shape)
print("Missing values after merges:", df.isna().sum().sum())


ambulance_count already matches resource_df aggregation for 100.0% of areas -> redundant, skip re-deriving
(5184, 45)
Missing values after merges: 0


(5184, 45)
Missing values after merges: 0


## 2. Target investigation & leakage check

`priority_score` (continuous) has a **0.9185 correlation with `resource_shortage`**, which is a
huge red flag: it suggests `priority_score` may have been formulaically generated rather than
independently labeled. We investigate two things before touching the model:

1. Is `priority_level` (our classification target) a direct, deterministic function of `priority_score`?
2. Is `priority_score` itself directly computable from a small number of other features in the dataset?


In [59]:
print(incident_df['priority_level'].value_counts())
print()
print(incident_df.groupby('priority_level')['priority_score'].agg(['min', 'max', 'mean', 'count']))


priority_level
MEDIUM    2410
HIGH      1702
LOW       1072
Name: count, dtype: int64

                  min    max       mean  count
priority_level                                
HIGH            50.03  73.51  62.154612   1702
LOW             13.46  25.00  21.695858   1072
MEDIUM          25.00  49.98  36.839071   2410


**Finding #1 — no `CRITICAL` incidents exist in this dataset.** Only `LOW`, `MEDIUM`, `HIGH` are
present. We keep the pipeline generic (it will support 4 classes if `CRITICAL` examples are added
later), but for now we train and evaluate on the 3 classes that actually exist, and we substitute
**`HIGH`-class recall** wherever the brief asks us to prioritize "recall for CRITICAL incidents" —
`HIGH` is the most urgent class currently observed.

In [60]:
def bucket(score):
    if score < 25:
        return 'LOW'
    elif score < 50:
        return 'MEDIUM'
    else:
        return 'HIGH'

test_level = incident_df['priority_score'].apply(bucket)
match_rate = (test_level == incident_df['priority_level']).mean()
print(f"priority_level matches a simple threshold bucket of priority_score for {match_rate:.2%} of rows")
mismatches = incident_df[test_level != incident_df['priority_level']]
print(f"{len(mismatches)} mismatch(es), all at the exact boundary (score == 25.0):")
mismatches[['priority_score', 'priority_level']]


priority_level matches a simple threshold bucket of priority_score for 99.98% of rows
1 mismatch(es), all at the exact boundary (score == 25.0):


,priority_score,priority_level
3951,25.0,LOW


**Finding #2 — `priority_level` is a synthetic label.** It is (with one boundary rounding
exception) a deterministic threshold rule applied to `priority_score`
(`<25 -> LOW`, `25–50 -> MEDIUM`, `>=50 -> HIGH`). This confirms the brief's concern: this is a
**synthetic-label problem**, not an independently-annotated target.

Now we check what `priority_score` itself is built from.

In [61]:
from sklearn.linear_model import LinearRegression

candidate_components = ['resource_shortage', 'severity_score', 'vulnerable_ratio']
X_probe = incident_df[candidate_components].values
y_probe = incident_df['priority_score'].values

lr_probe = LinearRegression().fit(X_probe, y_probe)
print(f"R^2 of priority_score ~ {candidate_components}: {lr_probe.score(X_probe, y_probe):.4f}")

lr_rs_only = LinearRegression().fit(incident_df[['resource_shortage']], y_probe)
print(f"R^2 of priority_score ~ resource_shortage alone: {lr_rs_only.score(incident_df[['resource_shortage']], y_probe):.4f}")


R^2 of priority_score ~ ['resource_shortage', 'severity_score', 'vulnerable_ratio']: 0.8729
R^2 of priority_score ~ resource_shortage alone: 0.8437


**Finding #3 — `resource_shortage` alone explains ~84% of the variance in `priority_score`**,
and `resource_shortage` + `severity_score` + `vulnerable_ratio` together explain ~87%. These three
engineered scores are clearly direct components of the formula used to build `priority_score`
(and therefore `priority_level`).

**Decision, per the brief:** since `resource_shortage` is directly used to calculate
`priority_score`, we run **two experiments**:

- **Experiment A** — include `resource_shortage`, `severity_score`, `vulnerable_ratio` (the formula components). This will look artificially strong because the model can partially "read off" the label formula.
- **Experiment B** — exclude `resource_shortage`, `severity_score`, `vulnerable_ratio`, and `priority_score` entirely, and predict priority only from **raw operational signals**: casualties, affected population, vulnerable groups, resource demand/availability/capacity. This is the leak-safe, deployable model.

We use `priority_level` as the classification target (not the raw `priority_score`), and treat
Experiment B as the model we'd actually ship.

## 3. Preprocessing

In [62]:
# Drop IDs / irrelevant columns. `district` has 780 unique values across 5,184 rows -- too
# high-cardinality to one-hot encode without overfitting/exploding dimensionality for a
# hackathon-scale dataset, so we keep `state` (36 categories) and drop `district`.
drop_cols = ['incident_id', 'disaster_id', 'area_id', 'data_status', 'district']
model_df = df.drop(columns=drop_cols)

TARGET = 'priority_level'
ALWAYS_DROP = ['priority_score']                                   # the score the label is thresholded from
FORMULA_COMPONENTS = ['resource_shortage', 'severity_score', 'vulnerable_ratio']  # direct inputs to priority_score

categorical_cols = ['state', 'disaster_type', 'disaster_subtype']
numeric_cols_base = [c for c in model_df.columns
                      if c not in categorical_cols + [TARGET] + ALWAYS_DROP + FORMULA_COMPONENTS]

print(f"{len(categorical_cols)} categorical columns:", categorical_cols)
print(f"{len(numeric_cols_base)} base numeric columns (Experiment B feature set)")
print("No missing values remaining:", model_df.isna().sum().sum() == 0)


3 categorical columns: ['state', 'disaster_type', 'disaster_subtype']
32 base numeric columns (Experiment B feature set)
No missing values remaining: True


In [63]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def build_pipeline(model, categorical_cols, numeric_cols, scale=False):
    """Bundles preprocessing + model into a single sklearn Pipeline.
    OneHotEncoder for categoricals; numeric features are scaled only for
    models that need it (e.g. Logistic Regression) -- tree models don't."""
    numeric_step = ('scaler', StandardScaler()) if scale else ('passthrough', 'passthrough')
    numeric_pipe = Pipeline([numeric_step]) if scale else 'passthrough'
    preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', numeric_pipe, numeric_cols),
    ])
    return Pipeline([('preprocess', preprocessor), ('model', model)])


## 4. Model training — Experiment A vs Experiment B

In [64]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix, recall_score

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAS_XGB = False

LABEL_TO_INT = {'LOW': 0, 'MEDIUM': 1, 'HIGH': 2}
INT_TO_LABEL = {v: k for k, v in LABEL_TO_INT.items()}

def run_experiment(name, numeric_cols):
    X = model_df[categorical_cols + numeric_cols]
    y = model_df[TARGET]

    # class balance check
    print(f"--- {name}: class distribution ---")
    print(y.value_counts(normalize=True).round(3))

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    if HAS_XGB:
        boosted_model = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                                       eval_metric='mlogloss', random_state=42)
    else:
        boosted_model = HistGradientBoostingClassifier(random_state=42)

    candidates = {
        'LogisticRegression': (LogisticRegression(max_iter=2000, class_weight='balanced'), True),
        'RandomForest': (RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                                  random_state=42, n_jobs=-1), False),
        'XGBoost' if HAS_XGB else 'HistGradientBoosting': (boosted_model, False),
    }

    results = {}
    for model_name, (mdl, scale) in candidates.items():
        pipe = build_pipeline(mdl, categorical_cols, numeric_cols, scale=scale)
        if HAS_XGB and model_name == 'XGBoost':
            # XGBoost's sklearn API wants integer-encoded labels
            pipe.fit(X_train, y_train.map(LABEL_TO_INT))
            y_pred = pd.Series(pipe.predict(X_test)).map(INT_TO_LABEL).values
        else:
            pipe.fit(X_train, y_train)
            y_pred = pipe.predict(X_test)

        macro_f1 = f1_score(y_test, y_pred, average='macro')
        acc = (y_pred == y_test.values).mean()
        results[model_name] = {'pipeline': pipe, 'macro_f1': macro_f1, 'accuracy': acc,
                                'y_test': y_test, 'y_pred': y_pred}
        print(f"  {model_name:20s} macro_f1={macro_f1:.4f}  accuracy={acc:.4f}")
    return results

print("=" * 70)
print("EXPERIMENT A -- includes resource_shortage / severity_score / vulnerable_ratio")
print("=" * 70)
results_A = run_experiment("Experiment A", numeric_cols_base + FORMULA_COMPONENTS)

print()
print("=" * 70)
print("EXPERIMENT B -- excludes formula components (leak-safe, deployable)")
print("=" * 70)
results_B = run_experiment("Experiment B", numeric_cols_base)


EXPERIMENT A -- includes resource_shortage / severity_score / vulnerable_ratio
--- Experiment A: class distribution ---
priority_level
MEDIUM    0.465
HIGH      0.328
LOW       0.207
Name: proportion, dtype: float64


C:\Users\DELL-7280\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


  LogisticRegression   macro_f1=0.9055  accuracy=0.9113
  RandomForest         macro_f1=0.9136  accuracy=0.9161
  HistGradientBoosting macro_f1=0.9423  accuracy=0.9412

EXPERIMENT B -- excludes formula components (leak-safe, deployable)
--- Experiment B: class distribution ---
priority_level
MEDIUM    0.465
HIGH      0.328
LOW       0.207
Name: proportion, dtype: float64


C:\Users\DELL-7280\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


  LogisticRegression   macro_f1=0.8540  accuracy=0.8602
  RandomForest         macro_f1=0.8916  accuracy=0.8949
  HistGradientBoosting macro_f1=0.8943  accuracy=0.8968


  LogisticRegression   macro_f1=0.9055  accuracy=0.9113


  RandomForest         macro_f1=0.9136  accuracy=0.9161


  XGBoost              macro_f1=0.9435  accuracy=0.9421

EXPERIMENT B -- excludes formula components (leak-safe, deployable)
--- Experiment B: class distribution ---
priority_level
MEDIUM    0.465
HIGH      0.328
LOW       0.207
Name: proportion, dtype: float64
  LogisticRegression   macro_f1=0.8540  accuracy=0.8602


  RandomForest         macro_f1=0.8916  accuracy=0.8949


  XGBoost              macro_f1=0.8979  accuracy=0.9007


## 5. Model comparison & evaluation

In [65]:
comparison_rows = []
for exp_name, results in [('A (with formula components)', results_A),
                           ('B (leak-safe)', results_B)]:
    for model_name, r in results.items():
        comparison_rows.append({'experiment': exp_name, 'model': model_name,
                                 'macro_f1': round(r['macro_f1'], 4),
                                 'accuracy': round(r['accuracy'], 4)})
comparison_df = pd.DataFrame(comparison_rows).sort_values('macro_f1', ascending=False)
comparison_df


,experiment,model,macro_f1,accuracy
2,A (with formula components),HistGradientBoosting,0.9423,0.9412
1,A (with formula components),RandomForest,0.9136,0.9161
0,A (with formula components),LogisticRegression,0.9055,0.9113
5,B (leak-safe),HistGradientBoosting,0.8943,0.8968
4,B (leak-safe),RandomForest,0.8916,0.8949
3,B (leak-safe),LogisticRegression,0.8540,0.8602


As expected, Experiment A scores higher — because it partially has access to the formula that
generated the label. That gap is itself evidence of leakage, not evidence that Experiment A is a
better *operational* model.

**We select the Experiment B model as the final model** (best Macro F1 among leak-safe models,
and the one we can honestly deploy), and pick the best algorithm within it by Macro F1 first,
then `HIGH`-class recall (our proxy for `CRITICAL`, since no `CRITICAL` incidents exist in this
dataset).

In [66]:
best_model_name = max(results_B, key=lambda m: results_B[m]['macro_f1'])
best = results_B[best_model_name]
print(f"Best leak-safe model: {best_model_name}")
print()
print(classification_report(best['y_test'], best['y_pred']))

cm = confusion_matrix(best['y_test'], best['y_pred'], labels=['LOW', 'MEDIUM', 'HIGH'])
print("Confusion matrix (rows=actual, cols=predicted), order = [LOW, MEDIUM, HIGH]:")
print(cm)

high_recall = recall_score(best['y_test'], best['y_pred'], labels=['HIGH'], average='macro')
print(f"\nHIGH-class recall (proxy for CRITICAL): {high_recall:.4f}")


Best leak-safe model: HistGradientBoosting

              precision    recall  f1-score   support

        HIGH       0.95      0.91      0.93       341
         LOW       0.84      0.89      0.86       214
      MEDIUM       0.89      0.89      0.89       482

    accuracy                           0.90      1037
   macro avg       0.89      0.90      0.89      1037
weighted avg       0.90      0.90      0.90      1037

Confusion matrix (rows=actual, cols=predicted), order = [LOW, MEDIUM, HIGH]:
[[191  23   0]
 [ 37 428  17]
 [  0  30 311]]

HIGH-class recall (proxy for CRITICAL): 0.9120


## 6. Feature importance

In [67]:
best_pipeline = best['pipeline']
preprocessor = best_pipeline.named_steps['preprocess']
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
all_feature_names = list(cat_feature_names) + numeric_cols_base

importances = best_pipeline.named_steps['model'].feature_importances_
importance_df = (pd.DataFrame({'feature': all_feature_names, 'importance': importances})
                  .sort_values('importance', ascending=False)
                  .head(15)
                  .reset_index(drop=True))
importance_df


AttributeError: 'HistGradientBoostingClassifier' object has no attribute 'feature_importances_'

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.barh(importance_df['feature'][::-1], importance_df['importance'][::-1])
plt.xlabel('Importance')
plt.title(f'Top 15 features -- {best_model_name} (Experiment B, leak-safe)')
plt.tight_layout()
plt.show()


## 7. Cross-validation

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

X_all = model_df[categorical_cols + numeric_cols_base]
y_all = model_df[TARGET]

if HAS_XGB:
    cv_model = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                              eval_metric='mlogloss', random_state=42)
    y_all_cv = y_all.map(LABEL_TO_INT)
else:
    cv_model = HistGradientBoostingClassifier(random_state=42)
    y_all_cv = y_all

cv_pipe = build_pipeline(cv_model, categorical_cols, numeric_cols_base, scale=False)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(cv_pipe, X_all, y_all_cv, cv=skf, scoring='f1_macro')

print("5-fold CV Macro F1 scores:", np.round(cv_scores, 4))
print(f"Mean: {cv_scores.mean():.4f}  Std: {cv_scores.std():.4f}")


## 8. Sample predictions & explanation

In [ ]:
def predict_priority(incident, pipeline=best_pipeline, categorical_cols=categorical_cols,
                      numeric_cols=numeric_cols_base, label_map=INT_TO_LABEL if HAS_XGB else None):
    """
    Predict operational priority for a single incident.

    Parameters
    ----------
    incident : dict or pd.DataFrame
        Incident features. Missing expected columns default to 0.

    Returns
    -------
    dict with predicted_priority, confidence, and full class probabilities.
    """
    row = pd.DataFrame([incident]) if isinstance(incident, dict) else incident.copy()
    for col in categorical_cols + numeric_cols:
        if col not in row.columns:
            row[col] = 0
    row = row[categorical_cols + numeric_cols]

    proba = pipeline.predict_proba(row)[0]
    classes = pipeline.named_steps['model'].classes_
    if label_map is not None:
        class_labels = [label_map[c] for c in classes]
    else:
        class_labels = list(classes)

    best_idx = int(np.argmax(proba))
    return {
        'predicted_priority': class_labels[best_idx],
        'confidence': round(float(proba[best_idx]), 3),
        'probabilities': {lab: round(float(p), 3) for lab, p in zip(class_labels, proba)},
    }


In [ ]:
def explain_priority(incident, pipeline=best_pipeline, importance_df=importance_df,
                      categorical_cols=categorical_cols, numeric_cols=numeric_cols_base,
                      label_map=INT_TO_LABEL if HAS_XGB else None, top_n=5):
    """
    Human-readable explanation for a prediction, based on the incident's own values
    for the model's globally most important features (NOT a causal claim -- these
    are the features the model relies on most, not proof of what caused the outcome).
    """
    result = predict_priority(incident, pipeline, categorical_cols, numeric_cols, label_map)
    row = pd.DataFrame([incident]) if isinstance(incident, dict) else incident.iloc[[0]]

    numeric_importance = importance_df[importance_df['feature'].isin(numeric_cols)].head(top_n)

    factors = []
    for _, r in numeric_importance.iterrows():
        feat = r['feature']
        val = row[feat].values[0] if feat in row.columns else None
        if val is not None:
            factors.append(f"- {feat.replace('_', ' ')}: {val}")

    lines = [f"Priority: {result['predicted_priority']} (confidence {result['confidence']:.0%})",
             "",
             "Main contributing factors (model's most important features for this incident,",
             "not a claim of what caused the outcome):"]
    lines += factors
    return "\n".join(lines)


In [ ]:
# demo on a few real incidents (features only, target excluded)
sample = model_df.drop(columns=[TARGET]).iloc[[0, 100, 2000]]

for i in range(len(sample)):
    incident = sample.iloc[[i]]
    print(explain_priority(incident))
    print()


## 9. Saving the model

In [ ]:
os.makedirs('models', exist_ok=True)

bundle = {
    'pipeline': best_pipeline,
    'model_name': best_model_name,
    'experiment': 'B (leak-safe)',
    'label_map': INT_TO_LABEL if HAS_XGB else None,
    'categorical_cols': categorical_cols,
    'numeric_cols': numeric_cols_base,
    'excluded_leakage_features': ALWAYS_DROP + FORMULA_COMPONENTS,
    'note': (
        "This model predicts OPERATIONAL priority (resource/response urgency), "
        "not the value of any individual life. priority_level in the source data is a "
        "deterministic threshold of priority_score, and priority_score is itself heavily "
        "driven by resource_shortage/severity_score/vulnerable_ratio -- this model "
        "deliberately excludes those to avoid re-learning the label formula, and instead "
        "predicts from raw operational signals (casualties, affected population, "
        "resource demand/availability/capacity)."
    ),
}

joblib.dump(bundle, 'models/resq_priority_model.pkl')
print("Saved: models/resq_priority_model.pkl")


### Summary

- **Target used:** `priority_level` (LOW / MEDIUM / HIGH — no `CRITICAL` incidents exist in this dataset yet).
- **Leakage check:** `priority_level` is a deterministic threshold of `priority_score`; `priority_score` is itself driven mostly by `resource_shortage` (+ `severity_score`, `vulnerable_ratio`). These were excluded from the final model (Experiment B).
- **Final model:** best leak-safe model selected by Macro F1, with `HIGH`-recall as a secondary tiebreaker (proxy for `CRITICAL`).
- **What ResQ AI does:** helps responders identify incidents where **delay may have greater operational consequences** — it ranks operational urgency from casualties, affected/vulnerable population, and resource strain. It does not, and should not, be used to judge whose life matters more.


In [ ]:
print("Target distribution:")
print(incident_df["priority_level"].value_counts())

print("\nModel:")
print(best_model_name)

print("\nPipeline steps:")
print(best_model_name.named_steps if hasattr(best_model_name, "named_steps") else "Not a pipeline")

In [68]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

y_pred = best_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

NameError: name 'X_test' is not defined

In [ ]:
[name for name in globals() if 'model' in name.lower()]

In [ ]:
[name for name in globals() if not name.startswith('_')]

In [69]:
print([x for x in globals() if 'test' in x.lower()])

['test_level', 'train_test_split']


In [70]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Features
X = model_df[categorical_cols + numeric_cols_base]

# Target
y = model_df[TARGET]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Prediction
y_pred = best_pipeline.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.896817743490839
Macro F1: 0.8942960573137307

Classification Report:
              precision    recall  f1-score   support

        HIGH       0.95      0.91      0.93       341
         LOW       0.84      0.89      0.86       214
      MEDIUM       0.89      0.89      0.89       482

    accuracy                           0.90      1037
   macro avg       0.89      0.90      0.89      1037
weighted avg       0.90      0.90      0.90      1037


Confusion Matrix:
[[311   0  30]
 [  0 191  23]
 [ 17  37 428]]


In [71]:
print("TARGET:")
print(TARGET)

print("\nCategorical columns:")
print(categorical_cols)

print("\nNumeric columns:")
print(numeric_cols_base)

print("\nAll model features:")
print(categorical_cols + numeric_cols_base)

TARGET:
priority_level

Categorical columns:
['state', 'disaster_type', 'disaster_subtype']

Numeric columns:
['population', 'people_affected', 'deaths', 'injured', 'critical_patients', 'children_affected', 'elderly_affected', 'hospital_count', 'ambulance_count', 'available_ambulances', 'rescue_team_count', 'available_rescue_teams', 'shelter_capacity', 'available_shelter_capacity', 'hospital_capacity', 'medical_supply', 'ambulance_demand', 'rescue_team_demand', 'shelter_demand', 'medical_supply_demand', 'households', 'male_population', 'female_population', 'urban_population', 'rural_population', 'children_population_0_6', 'elderly_population_60_plus', 'area_sq_km', 'population_density', 'resource_pct_available', 'resource_pct_needs_maintenance', 'resource_avg_capacity']

All model features:
['state', 'disaster_type', 'disaster_subtype', 'population', 'people_affected', 'deaths', 'injured', 'critical_patients', 'children_affected', 'elderly_affected', 'hospital_count', 'ambulance_count'

In [72]:
import joblib

joblib.dump(best_pipeline, "aapdasetu_priority_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [73]:
print("===== AAPDASETU MODEL PERFORMANCE =====")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.2%}")
print(f"Macro F1 : {f1_score(y_test, y_pred, average='macro'):.2%}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

===== AAPDASETU MODEL PERFORMANCE =====
Accuracy : 89.68%
Macro F1 : 89.43%

Classification Report:
              precision    recall  f1-score   support

        HIGH       0.95      0.91      0.93       341
         LOW       0.84      0.89      0.86       214
      MEDIUM       0.89      0.89      0.89       482

    accuracy                           0.90      1037
   macro avg       0.89      0.90      0.89      1037
weighted avg       0.90      0.90      0.90      1037

